In [29]:
import pandas as pd
import spacy
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Raz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
df = pd.read_csv("../../data/10_steam_games_cleaned_translated_and_no_description_data.csv")

In [22]:
# Cleaning the descriptions using regexes

# Finds a position in a word where there is an uppercase letter followed by a lowercase letter and puts a space there
df['cleaned_description'] = df['combined_description'].str.replace(r'(?=[A-Z][a-z])', ' ', regex=True)

# Lowercase everything
df['cleaned_description'] = df['cleaned_description'].str.lower()

# Replace everything that is not a lowercase letter or a space with a space
df['cleaned_description'] = df['cleaned_description'].str.replace(r'[^a-z\s]', ' ', regex=True)

# Remove single letter words
df['cleaned_description'] = df['cleaned_description'].str.replace(r'\b[a-z]\b', ' ', regex=True)

# Remove all the extra spaces
df['cleaned_description'] = df['cleaned_description'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [23]:
# Remove stop words ("the", "is", "in", "for", "on", etc.)

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join(
        word for word in text.split()
        if word not in stop_words
    )

df['cleaned_description'] = df['cleaned_description'].apply(remove_stopwords)

In [24]:
df.to_csv("../../data/11_steam_games_cleaned_descriptions.csv", index=False)

In [32]:
df = pd.read_csv("../../data/11_steam_games_cleaned_descriptions.csv")

# Fill the missing values with an empty string
df['cleaned_description'] = (df['cleaned_description'].fillna('').astype(str))

# Load the English model and disable the parser and ner components to make the program faster
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

# Store here the lemmatized descriptions
lemmatized_descriptions = []

# Get the lemmatized game descriptions in batches of 1000, using all the available CPU cores (from n_process=-1)
for game_description in nlp.pipe(df['cleaned_description'].astype(str), batch_size = 1000, n_process = -1):
    # Get the lemmatized word and put them back in sentences with spaces between
    lemmatized_description = ' '.join([word.lemma_ for word in game_description])
    lemmatized_descriptions.append(lemmatized_description)

# Put the lemmatized descriptions in the DataFrame
df['cleaned_description'] = lemmatized_descriptions

df.to_csv("../../data/12_steam_games_lemmatized_descriptions.csv", index=False)

In [34]:
# Displaying the cleaned description at a specific appid

pd.set_option('display.max_colwidth', None)

print(df.loc[df['appid'] == 730]['cleaned_description'])
print('\n')

0    two decade counter strike offer elite competitive experience one shape million player across globe next chapter cs story begin counter strike free upgrade cs go counter strike mark large technical leap counter strike history build source engine counter strike modernize realistic physically base render state art network upgrade community workshop tool addition classic objective focus gameplay counter strike pioneer counter strike feature new cs rating update premier mode global regional leaderboard upgrade overhauled map game change dynamic smoke grenade tick rate independent gameplay redesign visual effect audio item cs go move forward cs two decade counter strike offer elite competitive experience one shape million player across globe next chapter cs story begin counter strike
Name: cleaned_description, dtype: str




In [ ]:
df1 = pd.read_csv("../../data/11_steam_games_cleaned_descriptions.csv")
df2 = pd.read_csv("../../data/12_steam_games_lemmatized_descriptions.csv")

pd.set_option('display.max_colwidth', None)

In [38]:
appid = 961220

print(df1.loc[df['appid'] == appid]['cleaned_description'])
print("\n")
print(df2.loc[df['appid'] == appid]['cleaned_description'])

55508    congratulations incredibly lucky week long vacation new elite sanatorium purgatorium away city bustle everyday stress slopes pine forest hills small paradise purpose give unforgettable experience relaxing give opportunity make new interesting acquaintances indulge delicious dishes create comfortable conditions believe us spending time sanatorium become completely different person sure make rest ticket code sanatorium purgatorium visual novel main character ryota gets opportunity spend week unique place company six future friends immerse plot imbued story characters make various decisions sanatorium purgatorium provides excellent opportunity relax forgetting worries make new friends meet love life cleanse sinful soul
Name: cleaned_description, dtype: str


55508    congratulation incredibly lucky week long vacation new elite sanatorium purgatorium away city bustle everyday stress slope pine forest hill small paradise purpose give unforgettable experience relax give opportunity 

In [53]:
print(df2.loc[df2['cleaned_description'].isna(), 'appid'])

# drop games with no value at the cleaned description column
df2 = df2[~df2['appid'].isin([15188, 1441010])]

print("Successfully dropped games with no values at the cleaned description column")

print(df2.loc[df2['cleaned_description'].isna(), 'appid'])

df2.to_csv("../../data/12_steam_games_lemmatized_descriptions.csv", index=False)

Series([], Name: appid, dtype: int64)
Successfully dropped games with no values at the cleaned description column
Series([], Name: appid, dtype: int64)
